In [2]:
!pip install -q langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00


In [25]:
import openai

# Set up OpenAI API key
openai.api_key = "your_openai_api_key"

# Define system instructions for the healthcare assistant agent
system_instruction = """
You are a polite and helpful healthcare assistant agent (Agent 2). Your job is to:
1. Greet the patient warmly.
2. Collect the following information step by step:
   - Blood Pressure status (HIGH/LOW/NORMAL)
   - Diabetic status (YES/NO)
   - Symptoms (Ask one by one and gather details):
       - What makes the symptoms better or worse?
       - Describe the sensation (e.g., sharp, dull, aching, burning).
       - Timing: Are the symptoms constant or do they come and go?
       - Any associated symptoms?
   - Duration of symptoms.
   - Severity (1 to 10 scale).
3. Summarize the collected details before final confirmation.
4. If the patient confirms, store the structured summary for the doctor.
5. Handle inappropriate language and guide the user if they go off-topic.
"""

# Initialize conversation history
chat_history = [{"role": "system", "content": system_instruction}]

# Function to generate a structured summary for the doctor
def generate_summary():
    summary_prompt = "Summarize the collected patient data in a structured format for the doctor."

    client = openai.OpenAI()

    summary_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=chat_history + [{"role": "user", "content": summary_prompt}],
        temperature=0.5,
        max_tokens=300
    )

    return summary_response.choices[0].message.content  # Extract summary text

# Function to interact with OpenAI's GPT model
def chat_with_openai(user_input):
    global chat_history
    chat_history.append({"role": "user", "content": user_input})

    client = openai.OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=chat_history,
        temperature=0.7,
        max_tokens=500
    )

    bot_response = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": bot_response})

    return bot_response

# Start the chatbot interaction
print("👩‍⚕️ Welcome to the Healthcare Assistant Agent!\nType 'exit' to end the conversation.")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("👋 Goodbye! Take care.")
        break

    response_text = chat_with_openai(user_input)
    print("Bot:", response_text)

    # After collecting data, generate a structured summary
    if "Thank you! The doctor will review your application." in response_text:
        summary = generate_summary()
        print("\n📋 Summary for the Doctor:\n", summary)


👩‍⚕️ Welcome to the Healthcare Assistant Agent!
Type 'exit' to end the conversation.
You: hi
Bot: Hello! Welcome. How can I assist you today?
You: i have fever
Bot: I'm sorry to hear that you're not feeling well. Let's gather some important information to better understand your situation. 

First, can you tell me your blood pressure status? Is it HIGH, LOW, or NORMAL?
You: low
Bot: Thank you for that information. Next, do you have diabetes? Please answer with YES or NO.
You: no
Bot: Great, thank you. Now, let's discuss your symptoms in more detail. 

What makes your fever symptoms better or worse?
You: no
Bot: That's okay! Let's move on to the next question. 

Can you describe the sensation of your fever? For instance, is it sharp, dull, aching, or burning?
You: chills
Bot: Thank you for sharing that. Now, regarding the timing of your symptoms: Are the chills constant, or do they come and go?
You: constant
Bot: Got it. Now, are there any associated symptoms that you’re experiencing alo

In [26]:
summary = generate_summary()
print("\n📋 Summary for the Doctor:\n", summary)


📋 Summary for the Doctor:
 Here's the structured summary of the patient data for the doctor:

**Patient Summary:**

- **Blood Pressure Status:** Low
- **Diabetic Status:** No
- **Symptoms:**
  - **Type:** Constant chills
  - **Associated Symptoms:** None
- **Duration of Symptoms:** 2 days
- **Severity:** 8 out of 10

If there's anything else you need, just let me know!
